In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install lifelines -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 7.7 MB/s eta 0:00:00


In [3]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold

Define paths and constants

In [4]:
PROCESSED_DATA_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/"
EXP_1_OUTPUT_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/exp_1/"
MODEL_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Model/exp_1/"
RESULTS_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/exp_1/"

os.makedirs(EXP_1_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

TRAIN_PATH = EXP_1_OUTPUT_DIR + "hcc_exp1_train_selected.csv"
TEST_PATH = EXP_1_OUTPUT_DIR + "hcc_exp1_test_selected.csv"

MODEL_PATH = MODEL_DIR + "hcc_exp1_coxph_model.pkl"
CV_RESULTS_PATH = RESULTS_DIR + "hcc_exp1_coxph_cv_results.csv"
CV_DETAILS_PATH = RESULTS_DIR + "hcc_exp1_coxph_cv_fold_details.csv"

DURATION_COL = "OS_MONTHS"
EVENT_COL = "OS_EVENT"

PENALIZER_GRID = [0.01, 0.05, 0.1, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
L1_RATIO_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]

N_SPLITS = 5
RANDOM_STATE = 42


Load selected train and test data

In [5]:
train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)

train_data.shape, test_data.shape


((1085, 32), (272, 32))

In [7]:
test_data.head()

,SAMPLE_COVERAGE,TUMOR_SIZE,MUT_ALB,MUT_ARID1A,MUT_ARID1B,MUT_UPF1,MUT_WHSC1,MUT_ZFHX3,CNA_CDKN2Ap16INK4A,SAMPLE_TYPE_Metastasis,...,C_N_,C_N_Yes,METASTASIS_Yes,MAJOR_LIVER_RESECTION_No,RESECTION_MARGIN_R0,NEOADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_No,OS_MONTHS,OS_EVENT
0,913.0,6.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,28.980,1
1,560.0,13.7,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,5.952,1
2,592.0,4.5,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,9.984,0
3,642.0,6.7,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,23.652,1
4,634.0,4.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,36.696,0


Basic checks

In [6]:
print("Train missing values:", int(train_data.isna().sum().sum()))
print("Test missing values:", int(test_data.isna().sum().sum()))

print("\nTrain OS_EVENT distribution:")
print(train_data[EVENT_COL].value_counts())

print("\nTest OS_EVENT distribution:")
print(test_data[EVENT_COL].value_counts())

Train missing values: 0
Test missing values: 0

Train OS_EVENT distribution:
OS_EVENT
1    641
0    444
Name: count, dtype: int64

Test OS_EVENT distribution:
OS_EVENT
1    161
0    111
Name: count, dtype: int64


ID leakage guard

In [8]:
for id_col in ["PATIENT_ID", "SAMPLE_ID"]:
    assert id_col not in train_data.columns, f"{id_col} found in train_data"
    assert id_col not in test_data.columns, f"{id_col} found in test_data"

print("ID leakage guard passed.")


ID leakage guard passed.


Optional drop block

In [9]:
COLS_TO_DROP = []

if COLS_TO_DROP:
    train_data = train_data.drop(columns=COLS_TO_DROP)
    test_data = test_data.drop(columns=COLS_TO_DROP)

train_data.shape, test_data.shape


((1085, 32), (272, 32))

Stratified CV helper

In [10]:
def apply_cross_validation(df, penalty, l1_ratio, duration_col, event_col, n_splits=5, random_state=42):
    X = df.reset_index(drop=True)
    y = X[event_col].astype(int).values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_details = []
    c_index_scores = []

    for fold_idx, (train_index, val_index) in enumerate(skf.split(X, y), start=1):
        train_fold = X.iloc[train_index].copy()
        val_fold = X.iloc[val_index].copy()

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                cph = CoxPHFitter(penalizer=penalty, l1_ratio=l1_ratio)
                cph.fit(train_fold, duration_col=duration_col, event_col=event_col)

            val_predictions = cph.predict_partial_hazard(val_fold)
            c_index = concordance_index(
                val_fold[duration_col],
                -val_predictions,
                val_fold[event_col]
            )

            c_index_scores.append(float(c_index))
            fold_details.append({
                "penalizer": penalty,
                "l1_ratio": l1_ratio,
                "fold": fold_idx,
                "status": "success",
                "c_index": float(c_index),
                "error_message": ""
            })

        except Exception as e:
            fold_details.append({
                "penalizer": penalty,
                "l1_ratio": l1_ratio,
                "fold": fold_idx,
                "status": "failed",
                "c_index": np.nan,
                "error_message": str(e)
            })

    return c_index_scores, fold_details


Grid search

In [11]:
cv_results = []
cv_fold_details = []

for penalty in PENALIZER_GRID:
    for l1_ratio in L1_RATIO_GRID:
        scores, details = apply_cross_validation(
            df=train_data,
            penalty=penalty,
            l1_ratio=l1_ratio,
            duration_col=DURATION_COL,
            event_col=EVENT_COL,
            n_splits=N_SPLITS,
            random_state=RANDOM_STATE
        )

        cv_fold_details.extend(details)

        successful_folds = len(scores)
        failed_folds = N_SPLITS - successful_folds

        cv_results.append({
            "penalizer": penalty,
            "l1_ratio": l1_ratio,
            "successful_folds": successful_folds,
            "failed_folds": failed_folds,
            "mean_c_index": np.mean(scores) if successful_folds > 0 else np.nan,
            "std_c_index": np.std(scores) if successful_folds > 0 else np.nan,
            "status": "ok" if successful_folds > 0 else "all_failed"
        })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    ["mean_c_index", "successful_folds"],
    ascending=[False, False]
).reset_index(drop=True)

cv_fold_details_df = pd.DataFrame(cv_fold_details)

cv_results_df.head()


,penalizer,l1_ratio,successful_folds,failed_folds,mean_c_index,std_c_index,status
0,0.30,0.00,5,0,0.722310,0.008094,ok
1,0.05,0.25,5,0,0.722186,0.010003,ok
2,0.50,0.00,5,0,0.721790,0.008106,ok
3,0.10,0.00,5,0,0.721263,0.010648,ok
4,0.01,1.00,5,0,0.720863,0.010570,ok


Save CV diagnostics

In [12]:
cv_results_df.to_csv(CV_RESULTS_PATH, index=False)
cv_fold_details_df.to_csv(CV_DETAILS_PATH, index=False)

CV_RESULTS_PATH, CV_DETAILS_PATH


('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/exp_1/hcc_exp1_coxph_cv_results.csv',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/exp_1/hcc_exp1_coxph_cv_fold_details.csv')

Select best setting

In [13]:
valid_results = cv_results_df.dropna(subset=["mean_c_index"]).copy()
best_penalty = valid_results.loc[0, "penalizer"]
best_l1_ratio = valid_results.loc[0, "l1_ratio"]
best_cv_c_index = valid_results.loc[0, "mean_c_index"]

print("Best penalizer:", best_penalty)
print("Best l1_ratio:", best_l1_ratio)
print("Best mean validation C-index:", round(best_cv_c_index, 4))


Best penalizer: 0.3
Best l1_ratio: 0.0
Best mean validation C-index: 0.7223


Train final model

In [14]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cph_final = CoxPHFitter(penalizer=best_penalty, l1_ratio=best_l1_ratio)
    cph_final.fit(train_data, duration_col=DURATION_COL, event_col=EVENT_COL)

cph_final.print_summary()


<lifelines.CoxPHFitter: fitted with 1085 total observations, 444 right-censored observations>
             duration col = 'OS_MONTHS'
                event col = 'OS_EVENT'
                penalizer = 0.3
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 1085
number of events observed = 641
   partial log-likelihood = -3746.69
         time fit was run = 2026-07-02 11:30:43 UTC

---
                                          coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                 
SAMPLE_COVERAGE                           0.00      1.00      0.00           -0.00            0.00                1.00                1.00
TUMOR_SIZE                                0.02      1.02      0.01           -0.00            0.03                1.00                1.03
MUT_ALB                                  -0.06      0.94      0.36           -0.77            0.66                0.46                1.93
MUT_ARID1A                                0.15      1.16      0.09           -0.02            0.32                0.98                1.37
MUT_ARID1B                               -0.22      0.80      0.23           -0.68            0.24                0.51                1.27
MUT_UPF1                                 -0.58      0.56      0.44           -1.45            0.28                0.24                1.32
MUT_WHSC1                                -0.66      0.52      0.41           -1.47            0.15                0.23                1.16
MUT_ZFHX3                                -0.62      0.54      0.21           -1.04           -0.21                0.35                0.81
CNA_CDKN2Ap16INK4A                       -0.18      0.83      0.05           -0.29           -0.08                0.75                0.92
SAMPLE_TYPE_Metastasis                   -0.09      0.92      0.08           -0.24            0.06                0.79                1.07
PRIMARY_SITE_Liver                       -0.15      0.86      0.07           -0.29           -0.01                0.75                0.99
GENE_PANEL_IMPACT410                      0.05      1.05      0.09           -0.12            0.23                0.89                1.25
GENE_PANEL_IMPACT468                      0.03      1.03      0.08           -0.12            0.18                0.88                1.20
GENE_PANEL_IMPACT505                     -0.14      0.87      0.11           -0.35            0.08                0.70                1.08
PRED_CATEGORY_Biliary class               0.33      1.39      0.11            0.11            0.54                1.12                1.72
ETHNICITY_Unknown whether Spanish or not -0.26      0.77      0.26           -0.76            0.25                0.47                1.29
MULTIFOCAL_DISEASE_Yes                    0.06      1.06      0.07           -0.08            0.20                0.93                1.22
LVI_No                                   -0.17      0.85      0.11           -0.37            0.04                0.69                1.04
PNI_                                      0.14      1.15      0.09           -0.03            0.31                0.97                1.36
PNI_No                                   -0.15      0.86      0.10           -0.34            0.04                0.71                1.04
P_N_                                      0.03      1.03      0.08           -0.11            0.18                0.89                1.20
P_N_Negative                             -0.15      0.86      0.08           -0.31            0.01                0.73                1.01
C_N_                                      0.02      1.02      0.12           -0.20            0.25                0.81                1.28
C_N_Yes                                   0.23      1.25      0.08            0.08           

Training C-index

In [15]:
train_c_index = cph_final.concordance_index_
print(f"Training Concordance Index: {train_c_index:.4f}")


Training Concordance Index: 0.7318


Quick test check

In [16]:
test_predictions = cph_final.predict_partial_hazard(test_data)
test_c_index = concordance_index(
    test_data[DURATION_COL],
    -test_predictions,
    test_data[EVENT_COL]
)

print(f"Test Concordance Index: {test_c_index:.4f}")


Test Concordance Index: 0.7352


Coefficient summary

In [17]:
cox_summary = cph_final.summary[["coef", "exp(coef)", "p"]].sort_values("p")
cox_summary.head(15)


,coef,exp(coef),p
covariate,,,
METASTASIS_Yes,0.409933,1.506716,8.998668e-08
RESECTION_MARGIN_R0,-0.302029,0.739317,4.961202e-04
CNA_CDKN2Ap16INK4A,-0.182373,0.833291,5.537024e-04
C_N_Yes,0.225169,1.252534,2.962213e-03
PRED_CATEGORY_Biliary class,0.327763,1.387860,3.102964e-03
MUT_ZFHX3,-0.622602,0.536547,3.343600e-03
PALLIATIVE_SYSTEMIC_THERPAY_No,-0.215542,0.806104,7.198927e-03
PRIMARY_SITE_Liver,-0.149108,0.861476,3.989528e-02
ADJUVANT_SYSTEMIC_THERAPY_Yes,-0.183255,0.832556,5.456081e-02


Save model

In [18]:
with open(MODEL_PATH, "wb") as pickle_file:
    pickle.dump(cph_final, pickle_file)

MODEL_PATH


'/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Model/exp_1/hcc_exp1_coxph_model.pkl'

**CoxPH Summary**

The optimized `exp_1` CoxPH training notebook completed successfully on the selected survival feature set. Cross-validation chose `penalizer = 0.3` and `l1_ratio = 0.0`, with a best mean validation C-index of `0.7223`. That means the elastic-net extension did not improve over the pure ridge-style CoxPH fit for this branch, which is a useful result in itself because it tells us the simpler regularization setting remained the strongest.

The final fitted model stayed nicely stable between train and test: training C-index was `0.7318` and test C-index was `0.7352`. That close alignment suggests the model generalizes well on the held-out cohort and is not showing obvious overfitting.

The strongest signals were clinically sensible and consistent with the rest of the pipeline. Higher risk was associated with `METASTASIS_Yes`, `C_N_Yes`, and `PRED_CATEGORY_Biliary class`, while lower risk was associated with `RESECTION_MARGIN_R0`, `CNA_CDKN2Ap16INK4A`, `MUT_ZFHX3`, and `PRIMARY_SITE_Liver`. A few additional variables such as `ADJUVANT_SYSTEMIC_THERAPY_Yes`, `P_N_Negative`, `MUT_ARID1A`, `PNI_`, and `TUMOR_SIZE` appeared near the top but with weaker statistical support.

In short, the updated CoxPH notebook preserved the earlier strong test performance while improving the training workflow with broader regularization search and cleaner validation tracking. This gives us a solid classical baseline before we move into the DeepSurv optimization for `exp_1`.